# Blender Cloud Renderer (Eevee + Cycles)

Renders a `.blend` file on a Google Colab GPU using a chosen Blender version.

* **Input**: a config.json + `.blend` file in one Google Drive folder (see `config/config.json` layout).
* **Output**: rendered still image or frame-by-frame PNGs written back to your Drive `output/` folder.
* **Engine**: Cycles (GPU) or Eevee. **Mode**: still image, or animation (render each frame; combine to a video yourself later).

> To add a brand-new Blender version, edit the `BLENDER_DOWNLOADS` dict in cell 02 (or set `blender.custom_tar_url` in your config.json).


In [ ]:
# @title 1) Authorize your Google Drive
# - Mounts the Drive account authorized by the user.
# - Does not access the publisher's Drive.
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted at /content/drive')


In [ ]:
# @title 2) Locate or create your renderer workspace
import json
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive')
DEFAULT_WORKSPACE = DRIVE_ROOT / 'BlenderCloudRenderer'

# @param {"type":"string"}
workspace_path = ''  # @param {type:"string"}

DEFAULT_CONFIG = {
    'drive': {
        'folder_id': '',
        'blend_filename': 'scene.blend',
        'output_subfolder': 'my_first_render',
    },
    'blend': {'source': 'drive', 'url': '', 'drive_id': ''},
    'render': {
        'engine': 'cycles',
        'mode': 'still',
        'resolution_percentage': 100,
        'samples': 128,
        'frame_start': 1,
        'frame_end': 250,
        'frame_step': 1,
        'use_gpu': True,
        'file_format': 'PNG',
        'color_mode': 'RGBA',
        'device': 'GPU',
    },
    'blender': {'major_minor': '4.2', 'custom_tar_url': ''},
}

CFG_FOLDER = Path(workspace_path).expanduser() if workspace_path.strip() else DEFAULT_WORKSPACE
if not CFG_FOLDER.is_relative_to(DRIVE_ROOT):
    raise ValueError('workspace_path must be inside /content/drive/MyDrive')

CFG_FOLDER.mkdir(parents=True, exist_ok=True)
(CFG_FOLDER / 'blend_files').mkdir(exist_ok=True)
(CFG_FOLDER / 'output').mkdir(exist_ok=True)
config_path = CFG_FOLDER / 'config.json'
if not config_path.exists():
    with open(config_path, 'w', encoding='utf-8') as fh:
        json.dump(DEFAULT_CONFIG, fh, indent=2)
    print('Created a new workspace in:', CFG_FOLDER)
    print('Upload your .blend file to:', CFG_FOLDER / 'blend_files')

print("Selected config folder:", CFG_FOLDER)

with open(CFG_FOLDER / 'config.json', 'r', encoding='utf-8') as fh:
    CONFIG = json.load(fh)
print("Loaded config.")


In [ ]:
# @title 3) Review + override configuration
p = lambda d: d if d else '(not set)'
print("Engine      :", CONFIG['render']['engine'])
print("Mode        :", CONFIG['render']['mode'])
print("Samples     :", CONFIG['render'].get('samples'))
print("Frames      :", (CONFIG['render'].get('frame_start'), CONFIG['render'].get('frame_end'), CONFIG['render'].get('frame_step')))
print("Blender ver :", CONFIG['blender'].get('major_minor'))
print("Blend file  :", CONFIG['drive'].get('blend_filename'))

# - Optional live overrides.
# @param engine engine: ["cycles","blender_eevee","eevee_next"] = "cycles"
engine = "cycles"
# @param mode mode: ["still","animation"] = "still"
mode = "still"
# @param blender_version Blender version: ["3.6","4.0","4.1","4.2","4.3","4.4","4.5","5.0","5.1","5.2"] = "4.2"
blender_version = "4.2"

# - Apply overrides when defined.
CONFIG['render']['engine'] = engine
CONFIG['render']['mode'] = mode
CONFIG['blender']['major_minor'] = blender_version
print("Overrides applied (if any).")


In [ ]:
# @title 4) Install Blender + Render
import json, os, subprocess, tempfile, tarfile, urllib.request, shutil
from pathlib import Path

CFG_FOLDER = Path(CFG_FOLDER)
BLEND_DIR = CFG_FOLDER / 'blend_files'
BLEND_PATH = BLEND_DIR / CONFIG['drive']['blend_filename']
OUT_DIR = CFG_FOLDER / 'output' / CONFIG['drive'].get('output_subfolder', 'render')

if not BLEND_PATH.exists():
    print("WARNING: blend not found at", BLEND_PATH)
    print("Upload your .blend file into the 'blend_files' subfolder next to config.json and re-run.")
    raise SystemExit

OUT_DIR.mkdir(parents=True, exist_ok=True)

# - Blender version download map.
BLENDER_DOWNLOADS = {
    "3.6": "https://download.blender.org/release/Blender3.6/blender-3.6.12-linux-x64.tar.xz",
    "4.0": "https://download.blender.org/release/Blender4.0/blender-4.0.2-linux-x64.tar.xz",
    "4.1": "https://download.blender.org/release/Blender4.1/blender-4.1.1-linux-x64.tar.xz",
    "4.2": "https://download.blender.org/release/Blender4.2/blender-4.2.2-linux-x64.tar.xz",
    "4.3": "https://download.blender.org/release/Blender4.3/blender-4.3.2-linux-x64.tar.xz",
    "4.4": "https://download.blender.org/release/Blender4.4/blender-4.4.1-linux-x64.tar.xz",
    "4.5": "https://download.blender.org/release/Blender4.5/blender-4.5.13-linux-x64.tar.xz",
    "5.0": "https://download.blender.org/release/Blender5.0/blender-5.0.1-linux-x64.tar.xz",
    "5.1": "https://download.blender.org/release/Blender5.1/blender-5.1.2-linux-x64.tar.xz",
    "5.2": "https://download.blender.org/release/Blender5.2/blender-5.2.1-linux-x64.tar.xz",
}
custom_url = CONFIG.get('blender', {}).get('custom_tar_url', '')
mm = CONFIG['blender'].get('major_minor', '4.2')
url = custom_url or BLENDER_DOWNLOADS.get(mm)
if not url:
    raise SystemExit(f"No Blender {mm}; add to BLENDER_DOWNLOADS or set custom_tar_url")

version = Path(url).name.split('-')[1]
install_dir = Path('/content/blender') / version
blender_bin = install_dir / 'blender'

if not (install_dir / 'blender').exists():
    print(f"Downloading Blender {version} ...")
    tar = Path('/content') / f"blender-{version}.tar.xz"
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req) as r, open(tar, 'wb') as f:
        shutil.copyfileobj(r, f)
    print("Extracting ...")
    with tarfile.open(tar, 'r:xz') as t:
        base = t.getmembers()[0].name.split('/')[0]
        for m in t.getmembers():
            if m.name == base: continue
            m.name = str(Path(m.name).relative_to(base))
            t.extract(m, install_dir)
    tar.unlink()

# - Blender Python driver.
driver = r'''
import json, os
from pathlib import Path
import bpy
cfg = json.load(open(os.environ['CFG'], encoding='utf-8'))
rc = cfg['render']
blend = os.environ['BLEND']; out = os.environ['OUT']
bpy.ops.wm.open_mainfile(filepath=blend)
scene = bpy.context.scene; rd = scene.render
rd.resolution_percentage = int(rc.get('resolution_percentage', 100))
emap = {'cycles':'CYCLES','blender_eevee':'BLENDER_EEVEE','eevee_next':'BLENDER_EEVEE_NEXT'}
rd.engine = emap.get(rc['engine'], 'CYCLES')
Path(out).mkdir(parents=True, exist_ok=True)
fmt = rc.get('file_format', 'PNG')
rd.image_settings.file_format = fmt
if rc['engine'] == 'cycles':
    scene.cycles.samples = int(rc.get('samples', 128))
    prefs = bpy.context.preferences.addons['cycles'].preferences
    try: prefs.compute_device_type = 'CUDA' if rc.get('use_gpu', True) else 'NONE'
    except Exception: pass
    prefs.get_devices()
    for d in prefs.devices: d.use = rc.get('use_gpu', True)
    try: scene.cycles.device = 'GPU' if rc.get('use_gpu', True) else 'CPU'
    except Exception: pass
else:
    rd.eevee.taa_render_samples = int(rc.get('samples', 64))
if rc['mode'] == 'animation':
    fs, fe, fp = int(rc.get('frame_start',1)), int(rc.get('frame_end',scene.frame_end)), int(rc.get('frame_step',1))
    for f in range(fs, fe+1, fp):
        scene.frame_set(f)
        rd.filepath = f"{out}/frame_{f:05d}.{fmt.lower()}"
        bpy.ops.render.render(write_still=True)
        print('rendered frame', f)
else:
    scene.frame_set(int(rc.get('frame_start',1)))
    rd.filepath = f"{out}/render"
    bpy.ops.render.render(write_still=True)
    print('rendered still')
'''

with tempfile.NamedTemporaryFile('w', suffix='.py', delete=False) as tf:
    tf.write(driver)
    driver_path = tf.name

env = dict(os.environ)
env['CFG'] = str(CFG_FOLDER / 'config.json')
env['BLEND'] = str(BLEND_PATH)
env['OUT'] = str(OUT_DIR)

print("Rendering ... this can take a while.")
r = subprocess.run([str(blender_bin), '--background', '--python', driver_path], env=env)
if r.returncode != 0:
    raise SystemExit(f"Blender failed with exit code {r.returncode}")

print("Done. Outputs in:", OUT_DIR)
for f in sorted(OUT_DIR.iterdir()):
    print("  ", f.name)
